# 01 Market Data Exploration

Use this notebook to inspect quote/history quality for US equities and ETFs before promoting provider behavior into `services/api/app/market_data`.

Questions to answer:

- Does `yfinance` return stable quote and history fields for common MVP symbols?
- Which fields are missing or inconsistent?
- What shape should persisted `market_quotes` and provider error handling use?

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

API_ROOT = REPO_ROOT / "services" / "api"
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

DATA_ROOT = REPO_ROOT / "data"
REPORTS_ROOT = DATA_ROOT / "reports"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

REPO_ROOT

PosixPath('/Users/michaeldere/Workspace/aiml/micromarket')

In [2]:
import pandas as pd
import yfinance as yf

symbols = ["SPY", "QQQ", "AAPL", "MSFT", "NVDA"]
period = "3mo"
interval = "1d"

In [3]:
frames = []
for symbol in symbols:
    ticker = yf.Ticker(symbol)
    history = ticker.history(period=period, interval=interval, auto_adjust=False)
    if history.empty:
        frames.append({"symbol": symbol, "rows": 0, "first_date": None, "last_date": None, "missing_close": None})
        continue
    frames.append(
        {
            "symbol": symbol,
            "rows": len(history),
            "first_date": history.index.min(),
            "last_date": history.index.max(),
            "missing_close": int(history["Close"].isna().sum()),
            "last_close": float(history["Close"].iloc[-1]),
            "last_volume": int(history["Volume"].iloc[-1]),
        }
    )

quality = pd.DataFrame(frames)
quality

,symbol,rows,first_date,last_date,missing_close,last_close,last_volume
0,SPY,63,2026-01-26 00:00:00-05:00,2026-04-24 00:00:00-04:00,0,713.940002,45123600
1,QQQ,63,2026-01-26 00:00:00-05:00,2026-04-24 00:00:00-04:00,0,663.880005,45423700
2,AAPL,63,2026-01-26 00:00:00-05:00,2026-04-24 00:00:00-04:00,0,271.059998,38124500
3,MSFT,63,2026-01-26 00:00:00-05:00,2026-04-24 00:00:00-04:00,0,424.619995,27413900
4,NVDA,63,2026-01-26 00:00:00-05:00,2026-04-24 00:00:00-04:00,0,208.270004,213780100


In [4]:
report_path = REPORTS_ROOT / "market_data_quality_sample.csv"
quality.to_csv(report_path, index=False)
report_path

PosixPath('/Users/michaeldere/Workspace/aiml/micromarket/data/reports/market_data_quality_sample.csv')

## Promotion Notes

When this exploration settles, implement provider behavior in `services/api/app/market_data` and test it there. Keep notebook observations as local reports, not runtime dependencies.